# Thực nghiệm 1: ResNet (baseline CNN) cho Facial Expression Recognition

Notebook độc lập, chạy **ResNet-18 hoặc ResNet-50** (tiền huấn luyện ImageNet, thư viện `timm`) lần lượt trên **RAF-DB** và **FERPlus** với cùng một quy trình huấn luyện.

**Thiết lập Kaggle trước khi chạy**
- *Settings → Accelerator*: **GPU T4 x1** hoặc **P100** (ResNet không cần 2 GPU).
- *Settings → Internet*: **On** (để tải trọng số tiền huấn luyện).
- *Add Input*: dataset **RAF-DB** (dạng thư mục `train/1..7`, `test/1..7`) và dataset **FER2013 có file `fer2013.csv`** (cột `emotion, pixels, Usage`).
- Nhãn FERPlus (`fer2013new.csv`) được ô **0** tự tải từ GitHub của Microsoft.

**Cách dùng**: sửa ô **1. Cấu hình** (đường dẫn, model) → *Run All*. Thời gian ước tính với ResNet-18 trên T4: khoảng 1 giờ cho 2 dataset, 1 seed.

Kết quả lưu ở `/kaggle/working/results/<model>/`: `results.csv`, ma trận nhầm lẫn, đường cong huấn luyện, trọng số tốt nhất. Nếu notebook bị ngắt giữa chừng, chạy lại thì các run đã xong sẽ được **bỏ qua**.

## 0. Kiểm tra input và tải nhãn FERPlus
In ra đường dẫn thật của các dataset đã Add Input (dùng để điền ô Cấu hình nếu tự tìm không thành công).

In [ ]:
!ls /kaggle/input
!find /kaggle/input -maxdepth 4 -type d | head -40
!find /kaggle/input -maxdepth 5 -name "*.csv" | head -20

# Nhãn FERPlus của Microsoft (cần bật Internet)
!wget -q -O /kaggle/working/fer2013new.csv https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv
!head -2 /kaggle/working/fer2013new.csv

## 1. Cấu hình

In [ ]:
from pathlib import Path

# ---------------- Kiến trúc ----------------
MODEL_NAME = "resnet18"                      # "resnet18" hoặc "resnet50"
TIMM_NAMES = {"resnet18": "resnet18.a1_in1k", "resnet50": "resnet50.a1_in1k"}
LR_BY_MODEL = {"resnet18": 3e-4, "resnet50": 2e-4}

# ---------------- Thực nghiệm ----------------
DATASETS_TO_RUN = ["rafdb", "ferplus"]   # bỏ bớt nếu muốn chạy riêng từng dataset
SEEDS = [0]                              # bài báo: [0, 1, 2] để báo cáo mean ± std

# ---------------- Siêu tham số chung (giống nhau cho mọi kiến trúc sau này) ----------------
IMG_SIZE = 224
BATCH_SIZE = 64
LR = LR_BY_MODEL[MODEL_NAME]
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 2
LABEL_SMOOTHING = 0.1
# Xử lý mất cân bằng lớp: lấy mẫu lại theo CĂN BẬC HAI tần suất lớp (chỉ áp dụng cho tập train).
# Lớp hiếm được lấy lặp lại nhiều hơn, mỗi lần lại qua augmentation ngẫu nhiên khác nhau.
SAMPLING = "sqrt"
EARLY_STOP_PATIENCE = 10      # dừng nếu val accuracy không tăng sau N epoch (0 = tắt)
NUM_WORKERS = 4
USE_AMP = True                # mixed precision fp16

# ---------------- Dữ liệu: SỬA ĐƯỜNG DẪN CHO KHỚP VỚI INPUT CỦA BẠN ----------------
# type="folder": root/<train_dir>/<thư mục lớp>/*.jpg  (thư mục train/test có thể nằm sâu bên trong root, notebook tự tìm)
# class_dirs: tên thư mục của từng lớp, CÙNG THỨ TỰ với classes
DATASETS = {
    "rafdb": dict(
        type="folder",
        root="/kaggle/input/raf-db-dataset",
        train_dir="train", test_dir="test",
        class_dirs=["1", "2", "3", "4", "5", "6", "7"],
        classes=["surprise", "fear", "disgust", "happiness", "sadness", "anger", "neutral"],
        val_ratio=0.1,            # RAF-DB không có tập val -> tách 10% train (cố định)
        epochs=40,
    ),
    # FERPlus gốc = ảnh của fer2013.csv + nhãn mới của Microsoft (fer2013new.csv).
    # Nhãn = cảm xúc nhiều phiếu nhất trong 8 cảm xúc; bỏ ảnh mà phiếu 'unknown'/'NF' >= phiếu cao nhất.
    # Usage: Training -> train, PublicTest -> val, PrivateTest -> test (phân bố lớp tự nhiên).
    "ferplus": dict(
        type="ferplus_csv",
        fer2013_csv="auto",                                  # "auto" = tự tìm fer2013.csv trong /kaggle/input
        ferplus_csv="/kaggle/working/fer2013new.csv",        # do ô 0 tải về
        epochs=30,
    ),
}

OUT_DIR = Path("/kaggle/working/results") / f"{MODEL_NAME}_{SAMPLING}"   # thư mục riêng, không lẫn với kết quả cũ
SPLIT_SEED = 2026                          # cố định cách tách val cho mọi model -> so sánh công bằng
INPUT_ROOT = Path("/kaggle/input")         # nơi tự tìm dữ liệu khi đường dẫn khai báo không tồn tại
CACHE_DIR = Path("/tmp/fer_cache")         # ảnh FERPlus 48x48 giải nén từ CSV (không lưu vào output)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_NAME, "| lr:", LR, "| output:", OUT_DIR)

## 2. Thư viện

In [ ]:
import json, math, os, random, time, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
import timm

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Đọc dữ liệu
Mỗi dataset được chuyển về 3 bảng `train / val / test` gồm cột `path, label`.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_dir(root, name, max_depth=4):
    """Tìm thư mục tên `name` (không phân biệt hoa thường) bên dưới root, ưu tiên nông nhất."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Không tồn tại: {root}")
    level = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            if d.name.lower() == name.lower() and d != root:
                return d
            try:
                subs = [p for p in d.iterdir() if p.is_dir()]
            except PermissionError:
                continue
            if len(subs) < 200:          # không lặn vào thư mục chứa hàng nghìn thư mục con
                nxt += subs
        level = nxt
    raise FileNotFoundError(f"Không tìm thấy thư mục '{name}' trong {root}")


def list_images(d):
    out = []
    for dp, _, files in os.walk(d):
        out += [os.path.join(dp, f) for f in files if os.path.splitext(f)[1].lower() in IMG_EXTS]
    return sorted(out)


def load_folder_split(root, split_dir, class_dirs):
    base = find_dir(root, split_dir)
    present = {p.name.lower(): p for p in base.iterdir() if p.is_dir()}
    rows = []
    for label, cd in enumerate(class_dirs):
        if cd.lower() not in present:
            raise FileNotFoundError(f"Thiếu thư mục lớp '{cd}' trong {base}. Có: {sorted(present)}")
        rows += [(p, label) for p in list_images(present[cd.lower()])]
    return pd.DataFrame(rows, columns=["path", "label"])


def auto_find_folder_root(train_dir, test_dir, class_dirs, search_root, max_depth=5):
    """Tìm thư mục chứa cả <train_dir>/ và <test_dir>/, trong <train_dir>/ có đủ các thư mục lớp."""
    want = {c.lower() for c in class_dirs}
    for dp, dirs, _ in os.walk(search_root):
        if len(Path(dp).relative_to(search_root).parts) >= max_depth:
            dirs[:] = []
            continue
        low = {d.lower(): d for d in dirs}
        if train_dir.lower() in low and test_dir.lower() in low:
            sub = {s.lower() for s in os.listdir(os.path.join(dp, low[train_dir.lower()]))}
            if want <= sub:
                return Path(dp)
    return None


def find_csv(path, names, search_root):
    """Trả về path nếu tồn tại, nếu không thì tìm file có tên trong `names` dưới search_root."""
    if path != "auto" and Path(path).exists():
        return Path(path)
    for dp, _, files in os.walk(search_root):
        for n in names:
            if n in files:
                return Path(dp) / n
    raise FileNotFoundError(f"Không tìm thấy {names} (đường dẫn khai báo: {path}). "
                            f"Hãy Add Input dataset FER2013 có file fer2013.csv.")


FERPLUS_CLASSES = ["neutral", "happiness", "surprise", "sadness", "anger", "disgust", "fear", "contempt"]


def load_ferplus(fer2013_csv, ferplus_csv, cache_dir):
    fer_path = find_csv(fer2013_csv, ["fer2013.csv", "icml_face_data.csv"], INPUT_ROOT)
    new_path = find_csv(ferplus_csv, ["fer2013new.csv"], INPUT_ROOT if ferplus_csv == "auto" else Path("/kaggle/working"))
    print(f"  ảnh : {fer_path}\n  nhãn: {new_path}")
    fer = pd.read_csv(fer_path)
    fer.columns = [c.strip() for c in fer.columns]       # icml_face_data.csv có khoảng trắng trong tên cột
    new = pd.read_csv(new_path)
    if not {"pixels", "Usage"} <= set(fer.columns):
        raise ValueError(f"{fer_path.name} thiếu cột 'pixels'/'Usage' (cột hiện có: {list(fer.columns)}). "
                         "Bạn đang dùng bản FER2013 dạng thư mục ảnh? Cần bản có file CSV.")
    if len(fer) != len(new):
        raise ValueError(f"Số dòng không khớp: {fer_path.name} có {len(fer)}, fer2013new.csv có {len(new)} (cần 35887).")
    cols = {c.lower(): c for c in new.columns}
    votes = new[[cols[c] for c in FERPLUS_CLASSES]].to_numpy()
    junk = np.maximum(new[cols["unknown"]].to_numpy(), new[cols["nf"]].to_numpy())
    usage = {"Training": "train", "PublicTest": "val", "PrivateTest": "test"}
    img_dir = Path(cache_dir) / "ferplus"
    img_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, (px, use) in enumerate(zip(fer["pixels"], fer["Usage"].astype(str).str.strip())):
        top = votes[i].max()
        if top <= 0 or junk[i] >= top or use not in usage:
            continue
        f = img_dir / f"{i:05d}.png"
        if not f.exists():
            Image.fromarray(np.asarray(px.split(), dtype=np.uint8).reshape(48, 48)).save(f)
        rows.append((str(f), int(votes[i].argmax()), usage[use]))
    df = pd.DataFrame(rows, columns=["path", "label", "split"])
    parts = [df[df.split == s][["path", "label"]].reset_index(drop=True) for s in ("train", "val", "test")]
    return (*parts, list(FERPLUS_CLASSES))


def load_dataset(name):
    c = DATASETS[name]
    if c["type"] == "ferplus_csv":
        return load_ferplus(c["fer2013_csv"], c["ferplus_csv"], CACHE_DIR)
    if not Path(c["root"]).exists():
        found = auto_find_folder_root(c["train_dir"], c["test_dir"], c["class_dirs"], INPUT_ROOT)
        if found is None:
            raise FileNotFoundError(f"[{name}] '{c['root']}' không tồn tại và không tự tìm được thư mục chứa "
                                    f"'{c['train_dir']}/' + '{c['test_dir']}/'. Xem kết quả ô 0 và sửa 'root'.")
        print(f"  [{name}] '{c['root']}' không tồn tại -> dùng {found}")
        c["root"] = str(found)
    train = load_folder_split(c["root"], c["train_dir"], c["class_dirs"])
    test = load_folder_split(c["root"], c["test_dir"], c["class_dirs"])
    # RAF-DB không có val -> tách phân tầng từ train, cố định SPLIT_SEED
    train, val = train_test_split(train, test_size=c.get("val_ratio", 0.1), stratify=train["label"],
                                  random_state=SPLIT_SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True), c["classes"]


DATA = {}
for name in DATASETS_TO_RUN:
    tr, va, te, classes = load_dataset(name)
    DATA[name] = (tr, va, te, classes)
    counts = pd.DataFrame({s: d["label"].value_counts().sort_index() for s, d in (("train", tr), ("val", va), ("test", te))})
    counts.index = [classes[i] for i in counts.index]
    print(f"\n===== {name}: train {len(tr)} | val {len(va)} | test {len(te)}")
    display(counts.fillna(0).astype(int).T)

### Kiểm tra nhãn bằng mắt
Mỗi hàng là một lớp. **Nếu hàng "happiness" không phải mặt cười thì `class_dirs` đang sai thứ tự.**

In [ ]:
def show_samples(name, per_class=6, seed=0):
    tr, _, _, classes = DATA[name]
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(classes), per_class, figsize=(per_class * 1.3, len(classes) * 1.4))
    for c, cname in enumerate(classes):
        paths = tr[tr.label == c].path.to_numpy()
        pick = rng.choice(paths, size=min(per_class, len(paths)), replace=False)
        for k in range(per_class):
            ax = axes[c, k]
            ax.axis("off")
            if k < len(pick):
                ax.imshow(Image.open(pick[k]).convert("RGB"))
            if k == 0:
                ax.set_title(cname, fontsize=9, loc="left")
    fig.suptitle(name, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


for name in DATASETS_TO_RUN:
    show_samples(name)

## 4. Dataset, augmentation, model

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2)], p=0.5),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.25)),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


class FERDataset(Dataset):
    def __init__(self, df, tf):
        self.paths, self.labels, self.tf = df["path"].tolist(), df["label"].to_numpy(), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")      # ảnh xám FERPlus -> 3 kênh
        return self.tf(img), int(self.labels[i])


def sqrt_sample_weights(labels):
    """Trọng số từng ảnh = 1/sqrt(n_lớp)  ->  xác suất lấy một lớp tỉ lệ với sqrt(n_lớp)."""
    counts = np.bincount(labels)
    return 1.0 / np.sqrt(counts[labels])


def expected_class_share(labels):
    """Tỉ lệ (%) mỗi lớp trong một epoch: trước và sau khi lấy mẫu lại."""
    counts = np.bincount(labels)
    return 100 * counts / counts.sum(), 100 * np.sqrt(counts) / np.sqrt(counts).sum()


def make_loaders(tr, va, te, seed):
    g = torch.Generator().manual_seed(seed)
    ds_tr = FERDataset(tr, train_tf)
    # Lấy có hoàn lại, số mẫu mỗi epoch = kích thước tập train (thời gian 1 epoch không đổi)
    w = torch.as_tensor(sqrt_sample_weights(ds_tr.labels), dtype=torch.double)
    sampler = WeightedRandomSampler(w, num_samples=len(ds_tr), replacement=True, generator=g)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    return (DataLoader(ds_tr, BATCH_SIZE, sampler=sampler, drop_last=True, **kw),
            DataLoader(FERDataset(va, eval_tf), BATCH_SIZE * 2, shuffle=False, **kw),
            DataLoader(FERDataset(te, eval_tf), BATCH_SIZE * 2, shuffle=False, **kw))


def build_model(num_classes):
    return timm.create_model(TIMM_NAMES[MODEL_NAME], pretrained=True, num_classes=num_classes)


def model_cost(model):
    """Số tham số (M), GMACs/ảnh (= FLOPs/2, tương đương 'FLOPs' trong đa số bài FER), FPS (batch 1)."""
    params = sum(p.numel() for p in model.parameters()) / 1e6
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(x)
        gmacs = fc.get_total_flops() / 2e9
    except Exception:
        gmacs = float("nan")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
        for _ in range(20):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(100):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return params, gmacs, 100 / (time.perf_counter() - t0)


m = build_model(7).to(DEVICE)
print("%s: %.2fM params, %.2f GMACs, %.0f FPS" % (MODEL_NAME, *model_cost(m)))
del m

## 5. Huấn luyện và đánh giá

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def metrics(y, p, k):
    cm = confusion_matrix(y, p, labels=list(range(k)))
    per_class = np.diag(cm) / np.maximum(cm.sum(1), 1)
    return {"acc": 100 * (y == p).mean(), "mca": 100 * per_class.mean(),
            "f1": 100 * f1_score(y, p, average="macro", labels=list(range(k)), zero_division=0),
            "per_class": (100 * per_class).round(2).tolist(), "cm": cm}


@torch.no_grad()
def evaluate(model, loader, k):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
            out = model(x.to(DEVICE, non_blocking=True))
        ps.append(out.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    return metrics(np.concatenate(ys), np.concatenate(ps), k)


def train_one_run(name, seed):
    tr, va, te, classes = DATA[name]
    cfg, k = DATASETS[name], len(classes)
    epochs = cfg["epochs"]
    seed_everything(seed)
    train_dl, val_dl, test_dl = make_loaders(tr, va, te, seed)
    before, after = expected_class_share(tr["label"].to_numpy())
    print("  tỉ lệ lớp/epoch (%) trước -> sau lấy mẫu lại:  " +
          ", ".join(f"{c} {b:.1f}->{a:.1f}" for c, b, a in zip(classes, before, after)))
    model = build_model(k).to(DEVICE)

    # AdamW; không weight decay cho bias và BatchNorm
    decay = [p for n, p in model.named_parameters() if p.ndim > 1]
    no_decay = [p for n, p in model.named_parameters() if p.ndim <= 1]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                             {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    total, warm = epochs * len(train_dl), WARMUP_EPOCHS * len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    best_path = OUT_DIR / f"{name}_s{seed}_best.pt"
    best_acc, best_ep, bad, hist = -1, 0, 0, []
    t_start = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        t0, loss_sum, correct, n = time.time(), 0.0, 0, 0
        for x, y in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
                out = model(x)
            loss = criterion(out.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
        v = evaluate(model, val_dl, k)
        hist.append({"epoch": ep, "train_loss": loss_sum / n, "train_acc": 100 * correct / n,
                     "val_acc": v["acc"], "val_f1": v["f1"], "lr": opt.param_groups[0]["lr"]})
        if v["acc"] > best_acc:
            best_acc, best_ep, bad = v["acc"], ep, 0
            torch.save(model.state_dict(), best_path)
        else:
            bad += 1
        print(f"  ep {ep:2d}/{epochs}  loss {loss_sum / n:.4f}  train {100 * correct / n:5.2f}  "
              f"val {v['acc']:5.2f} (f1 {v['f1']:5.2f})  best@{best_ep}  {time.time() - t0:.0f}s")
        if EARLY_STOP_PATIENCE and bad >= EARLY_STOP_PATIENCE:
            print(f"  early stopping: val không tăng trong {EARLY_STOP_PATIENCE} epoch")
            break

    # Đánh giá test DUY NHẤT MỘT LẦN, với trọng số có val tốt nhất (đúng phương pháp)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    t = evaluate(model, test_dl, k)
    params, gmacs, fps = model_cost(model)
    pd.DataFrame(hist).to_csv(OUT_DIR / f"{name}_s{seed}_history.csv", index=False)
    plot_run(name, seed, hist, t["cm"], classes)
    return {"model": MODEL_NAME, "sampling": SAMPLING, "dataset": name, "seed": seed, "test_acc": round(t["acc"], 2),
            "test_mca": round(t["mca"], 2), "test_f1": round(t["f1"], 2), "best_val_acc": round(best_acc, 2),
            "best_epoch": best_ep, "epochs_run": len(hist), "n_train": len(tr), "n_val": len(va), "n_test": len(te),
            "params_M": round(params, 2), "gmacs": round(gmacs, 2), "fps": round(fps, 1),
            "train_min": round((time.time() - t_start) / 60, 1),
            "per_class_acc": json.dumps(dict(zip(classes, t["per_class"])))}


def plot_run(name, seed, hist, cm, classes):
    h = pd.DataFrame(hist)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    a1.plot(h.epoch, h.train_acc, color="#2a78d6", lw=2, label="train acc")
    a1.plot(h.epoch, h.val_acc, color="#eb6834", lw=2, label="val acc")
    a1.set_xlabel("epoch"); a1.set_ylabel("accuracy (%)"); a1.legend(frameon=False)
    a1.set_title(f"{MODEL_NAME} / {name} / seed {seed}", loc="left", fontsize=10)
    a1.spines[["top", "right"]].set_visible(False); a1.grid(axis="y", color="#e6e6e3")
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    a2.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    a2.set_xticks(range(len(classes)), classes, rotation=45, ha="right", fontsize=8)
    a2.set_yticks(range(len(classes)), classes, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            a2.text(j, i, f"{100 * norm[i, j]:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.55 else "#222")
    a2.set_xlabel("dự đoán"); a2.set_ylabel("thực tế"); a2.set_title("confusion matrix (test, %)", loc="left", fontsize=10)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"{name}_s{seed}_curves_cm.png", dpi=120)
    plt.show()

## 6. Chạy thực nghiệm
Các run đã có trong `results.csv` sẽ được bỏ qua, nên có thể chạy lại ô này sau khi bị ngắt.

In [ ]:
RESULTS = OUT_DIR / "results.csv"
done = set()
if RESULTS.exists():
    old = pd.read_csv(RESULTS)
    done = set(zip(old.dataset, old.seed))

for name in DATASETS_TO_RUN:
    for seed in SEEDS:
        if (name, seed) in done:
            print(f"[skip] {name} seed {seed} đã có kết quả")
            continue
        print(f"\n########## {MODEL_NAME} | {name} | seed {seed} ##########")
        row = train_one_run(name, seed)
        pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not RESULTS.exists(), index=False)
        print(f"==> TEST  acc {row['test_acc']:.2f}  |  mean class acc {row['test_mca']:.2f}  |  macro-F1 {row['test_f1']:.2f}")
        torch.cuda.empty_cache()

## 7. Tổng hợp kết quả

In [ ]:
res = pd.read_csv(RESULTS)
display(res.drop(columns=["per_class_acc"]))

def pm(s):
    return f"{s.mean():.2f} ± {s.std():.2f}" if len(s) > 1 else f"{s.mean():.2f}"

summary = res.groupby("dataset").agg(seeds=("seed", "nunique"), acc=("test_acc", pm), mca=("test_mca", pm),
                                     f1=("test_f1", pm), params_M=("params_M", "first"), gmacs=("gmacs", "first"),
                                     fps=("fps", "median")).reindex([d for d in DATASETS_TO_RUN if d in set(res.dataset)])
summary.insert(0, "model", MODEL_NAME)
summary.to_csv(OUT_DIR / "summary.csv")
display(summary)

print("\nĐộ chính xác từng lớp (seed đầu tiên):")
for _, r in res.drop_duplicates("dataset").iterrows():
    print(f"  {r.dataset:10s}", json.loads(r.per_class_acc))

## 8. Trực quan hóa kết quả
Mọi biểu đồ được lưu (PNG, 200 dpi) vào `OUT_DIR/figures/` để đưa vào bài báo. Phần này đọc `results.csv`, các file `*_history.csv` và **trọng số tốt nhất đã lưu** (`*_best.pt`), nên chạy được cả với các run đã xong từ trước mà không cần huấn luyện lại.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Mục 8 tự đọc lại kết quả, không phụ thuộc vào việc ô mục 6-7 đã chạy trong phiên này hay chưa.
# Chỉ cần đã chạy các ô 1-5 (cấu hình, thư viện, đọc dữ liệu, model, hàm huấn luyện).
_missing = [v for v in ("OUT_DIR", "DATA", "build_model", "FERDataset", "eval_tf") if v not in globals()]
if _missing:
    raise RuntimeError(f"Chưa có {_missing}: hãy chạy các ô từ mục 1 đến mục 5 trước khi chạy mục 8.")
RESULTS = OUT_DIR / "results.csv"
if not RESULTS.exists():
    raise FileNotFoundError(f"Chưa có {RESULTS}: hãy chạy mục 6 (huấn luyện) trước.")
res = pd.read_csv(RESULTS)
res = res[res.dataset.isin(DATASETS_TO_RUN)]

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
# Bảng màu phân loại cố định thứ tự (đã kiểm tra cho người mù màu); chữ dùng màu mực trung tính
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e3"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.titlelocation": "left", "axes.titlesize": 11, "axes.titlecolor": INK,
                     "legend.frameon": False, "figure.dpi": 100})


def style_grid(ax):
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def save(fig, fname):
    fig.savefig(FIG_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()


def available(name):
    return sorted(res.loc[res.dataset == name, "seed"].unique())


# ---------------------------------------------------------------- dự đoán lại tập test từ trọng số tốt nhất
@torch.no_grad()
def predict_test(name, seed):
    tr, va, te, classes = DATA[name]
    path = OUT_DIR / f"{name}_s{seed}_best.pt"
    if not path.exists():
        print(f"  thiếu {path.name}, bỏ qua"); return None
    model = build_model(len(classes)).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    dl = DataLoader(FERDataset(te, eval_tf), BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)
    probs = []
    for x, _ in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
            probs.append(model(x.to(DEVICE)).float().softmax(1).cpu().numpy())
    del model; torch.cuda.empty_cache()
    probs = np.concatenate(probs)
    return {"y": te["label"].to_numpy(), "p": probs.argmax(1), "prob": probs, "paths": te["path"].to_numpy()}


PRED = {}
for name in DATASETS_TO_RUN:
    for seed in available(name):
        out = predict_test(name, seed)
        if out is not None:
            PRED[(name, seed)] = out
            print(f"  {name} seed {seed}: test acc {100 * (out['y'] == out['p']).mean():.2f}% (khớp results.csv)")

### 8.1 Đường cong huấn luyện
Loss train, accuracy train/val và macro-F1 trên val theo epoch (trung bình các seed; vùng mờ = min–max giữa các seed). Đường đứt nét là epoch được chọn.

In [ ]:
def plot_curves(name):
    hs = []
    for seed in available(name):
        f = OUT_DIR / f"{name}_s{seed}_history.csv"
        if f.exists():
            hs.append(pd.read_csv(f).assign(seed=seed))
    if not hs:
        return
    h = pd.concat(hs)
    agg = h.groupby("epoch").agg(["mean", "min", "max"])
    best = res.loc[res.dataset == name, "best_epoch"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    panels = [("Loss (train)", [("train_loss", "train", PALETTE[0])], "loss"),
              ("Accuracy", [("train_acc", "train", PALETTE[0]), ("val_acc", "val", PALETTE[1])], "accuracy (%)"),
              ("Macro-F1 (val)", [("val_f1", "val", PALETTE[1])], "macro-F1 (%)")]
    for ax, (title, series, ylab) in zip(axes, panels):
        for col, lab, color in series:
            ax.plot(agg.index, agg[(col, "mean")], color=color, lw=2, label=lab)
            if h.seed.nunique() > 1:
                ax.fill_between(agg.index, agg[(col, "min")], agg[(col, "max")], color=color, alpha=0.15, lw=0)
        for b in best:
            ax.axvline(b, color=MUTED, ls="--", lw=1)
        ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel(ylab); style_grid(ax)
        if len(series) > 1:
            ax.legend(loc="lower right")
    fig.suptitle(f"{MODEL_NAME} / {name}  ({h.seed.nunique()} seed)", x=0.01, ha="left", color=INK)
    fig.tight_layout()
    save(fig, f"curves_{name}.png")


for name in DATASETS_TO_RUN:
    plot_curves(name)

### 8.2 Ma trận nhầm lẫn
Trái: số lượng ảnh; phải: chuẩn hóa theo hàng (= recall từng lớp, %). Cộng dồn qua các seed nếu có nhiều seed.

In [ ]:
def plot_confusion(name):
    keys = [k for k in PRED if k[0] == name]
    if not keys:
        return
    classes = DATA[name][3]
    k = len(classes)
    cm = sum(confusion_matrix(PRED[x]["y"], PRED[x]["p"], labels=list(range(k))) for x in keys)
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(2 * (1.2 + 0.72 * k) + 1, 1.0 + 0.72 * k))
    for ax, mat, fmt, title in ((axes[0], cm, "{:.0f}", "số lượng"), (axes[1], norm * 100, "{:.0f}", "% theo hàng")):
        scale = mat / mat.max() if title == "số lượng" else mat / 100
        ax.imshow(scale, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(k), classes, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(k), classes, fontsize=9)
        ax.set_xlabel("dự đoán"); ax.set_ylabel("thực tế")
        ax.spines[:].set_visible(False)
        for i in range(k):
            for j in range(k):
                ax.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if scale[i, j] > 0.55 else INK)
        ax.set_title(f"{name}: {title}")
    fig.tight_layout()
    save(fig, f"confusion_{name}.png")


for name in DATASETS_TO_RUN:
    plot_confusion(name)

### 8.3 Precision / Recall / F1 từng lớp
Trung bình ± độ lệch chuẩn qua các seed. `n_test` = số ảnh test của lớp: lớp có rất ít ảnh thì chỉ số dao động mạnh.

In [ ]:
def per_class_report(name):
    keys = [k for k in PRED if k[0] == name]
    if not keys:
        return None
    classes = DATA[name][3]
    k = len(classes)
    rows = []
    for key in keys:
        p, r, f, n = precision_recall_fscore_support(PRED[key]["y"], PRED[key]["p"], labels=list(range(k)), zero_division=0)
        rows.append(pd.DataFrame({"class": classes, "precision": 100 * p, "recall": 100 * r, "f1": 100 * f,
                                  "n_test": n, "seed": key[1]}))
    df = pd.concat(rows)
    rep = df.groupby("class", sort=False).agg(precision=("precision", "mean"), precision_std=("precision", "std"),
                                              recall=("recall", "mean"), recall_std=("recall", "std"),
                                              f1=("f1", "mean"), f1_std=("f1", "std"), n_test=("n_test", "first"))
    rep.round(2).to_csv(OUT_DIR / f"per_class_{name}.csv")
    return rep


def plot_per_class(name, rep):
    x = np.arange(len(rep))
    w = 0.26
    fig, ax = plt.subplots(figsize=(1.3 * len(rep) + 2, 4.2))
    for i, (m, lab) in enumerate((("precision", "precision"), ("recall", "recall (= acc lớp)"), ("f1", "F1"))):
        err = rep[f"{m}_std"].fillna(0).to_numpy()
        ax.bar(x + (i - 1) * w, rep[m], width=w - 0.03, color=PALETTE[i], label=lab,
               yerr=err if err.any() else None, error_kw=dict(ecolor=INK, elinewidth=1, capsize=2))
    ax.set_xticks(x, [f"{c}\n(n={int(n)})" for c, n in zip(rep.index, rep.n_test)], fontsize=9)
    ax.set_ylim(0, 105); ax.set_ylabel("%"); style_grid(ax)
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.12))
    ax.set_title(f"{MODEL_NAME} / {name}: chỉ số từng lớp trên tập test", pad=28)
    fig.tight_layout()
    save(fig, f"per_class_{name}.png")


for name in DATASETS_TO_RUN:
    rep = per_class_report(name)
    if rep is not None:
        print(f"\n===== {name}")
        display(rep.round(2))
        plot_per_class(name, rep)

### 8.4 So sánh tổng hợp giữa các dataset

In [ ]:
def plot_overall():
    metrics_ = [("test_acc", "accuracy"), ("test_mca", "mean class acc"), ("test_f1", "macro-F1")]
    names = [d for d in DATASETS_TO_RUN if d in set(res.dataset)]
    g = res.groupby("dataset")
    x = np.arange(len(names))
    w = 0.26
    fig, ax = plt.subplots(figsize=(2.4 * len(names) + 2.5, 4))
    for i, (col, lab) in enumerate(metrics_):
        mean = g[col].mean().reindex(names).to_numpy()
        std = g[col].std().reindex(names).fillna(0).to_numpy()
        bars = ax.bar(x + (i - 1) * w, mean, width=w - 0.03, color=PALETTE[i], label=lab,
                      yerr=std if std.any() else None, error_kw=dict(ecolor=INK, elinewidth=1, capsize=3))
        for b, v, e in zip(bars, mean, std):
            ax.text(b.get_x() + b.get_width() / 2, v + e + 1.2, f"{v:.1f}", ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_xticks(x, names)
    ax.set_ylim(0, 105); ax.set_ylabel("%"); style_grid(ax)
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.13))
    ax.set_title(f"{MODEL_NAME}: kết quả trên tập test (mean ± std qua {res.seed.nunique()} seed)", pad=28)
    fig.tight_layout()
    save(fig, "overall.png")


plot_overall()

### 8.5 Ví dụ dự đoán sai
Ảnh test bị dự đoán sai (seed đầu tiên), kèm nhãn thật, nhãn dự đoán và độ tin cậy. Hữu ích cho phần phân tích lỗi trong bài.

In [ ]:
def plot_errors(name, n=16, seed_pick=0):
    keys = sorted(k for k in PRED if k[0] == name)
    if not keys:
        return
    d = PRED[keys[0]]
    classes = DATA[name][3]
    wrong = np.where(d["y"] != d["p"])[0]
    if len(wrong) == 0:
        print(f"{name}: không có ảnh dự đoán sai"); return
    rng = np.random.default_rng(seed_pick)
    pick = rng.choice(wrong, size=min(n, len(wrong)), replace=False)
    cols = 8
    rows = int(np.ceil(len(pick) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.9, rows * 2.2 + 0.4), gridspec_kw={"hspace": 0.32})
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, i in zip(np.atleast_1d(axes).ravel(), pick):
        ax.imshow(Image.open(d["paths"][i]).convert("RGB"))
        ax.set_title(f"thật: {classes[d['y'][i]]}\nđoán: {classes[d['p'][i]]} ({100 * d['prob'][i].max():.0f}%)",
                     fontsize=8, loc="center", color=INK)
    fig.suptitle(f"{name}: {len(wrong)}/{len(d['y'])} ảnh test bị dự đoán sai (seed {keys[0][1]}), hiển thị {len(pick)}",
                 x=0.01, ha="left", color=INK, fontsize=10)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=1 - 0.75 / (rows * 2.2 + 0.4))
    save(fig, f"errors_{name}.png")


for name in DATASETS_TO_RUN:
    plot_errors(name)

print("\nĐã lưu:", sorted(p.name for p in FIG_DIR.glob("*.png")))

## Ghi chú cho bài báo
- **Quy trình**: AdamW (wd 0.05), warmup 2 epoch + cosine, label smoothing 0.1, AMP, ảnh 224×224, augmentation (random resized crop, lật ngang, xoay ±10°, color jitter, random erasing). Giữ **nguyên** các thiết lập này cho mọi kiến trúc tiếp theo để so sánh công bằng.
- **Chọn mô hình theo val, đánh giá test một lần**: RAF-DB không có val chính thức nên tách cố định 10% train (`SPLIT_SEED = 2026`); FERPlus dùng PublicTest làm val và PrivateTest làm test. Nhãn FERPlus lấy theo đa số phiếu trên 8 cảm xúc (bỏ ảnh có phiếu 'unknown'/'not face' bằng hoặc cao hơn phiếu cao nhất); ảnh gốc 48×48 ảnh xám. Ghi rõ điều này trong bài.
- **Mất cân bằng lớp**: tập train được lấy mẫu lại có hoàn lại với xác suất mỗi lớp tỉ lệ với căn bậc hai số ảnh của lớp đó (square-root resampling), số mẫu mỗi epoch bằng kích thước tập train; ảnh lặp lại vẫn qua augmentation ngẫu nhiên nên mỗi lần là một biến thể khác. Val/test giữ nguyên.
- Báo cáo thêm **mean class accuracy** và **macro-F1** vì dữ liệu mất cân bằng lớp.
- Sau khi chạy xong `resnet18`, đổi `MODEL_NAME = "resnet50"` và chạy lại; kết quả lưu vào thư mục riêng.